## 0. 기본 세팅

In [ ]:
import os
print(os.getcwd())

/content


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 현재 작업 디렉토리를 'model' 폴더로 변경
# 'KOPIS 공모전' 폴더가 내 드라이브 바로 아래에 있다고 가정합니다.
os.chdir('/content/drive/MyDrive/KOPIS 공모전/model')

# 현재 작업 디렉토리 확인 (이제 'model' 폴더가 되어야 합니다)
print(os.getcwd())

/content/drive/MyDrive/KOPIS 공모전/model


## 1. 모델 생성을 위한 라이브러리 구축

In [ ]:
!pip uninstall torch -y
!pip uninstall torch-geometric -y
!pip install torch torch-geometric

Found existing installation: torch 2.8.0+cu126
Uninstalling torch-2.8.0+cu126:
  Successfully uninstalled torch-2.8.0+cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 4.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import torch.optim as optim
import re
from torch_geometric.utils import to_undirected, add_self_loops
from sklearn.metrics import f1_score, hamming_loss
import datetime

# =========================
# 1. 장치 설정
# =========================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"현재 사용 가능한 장치: {device}")

현재 사용 가능한 장치: cuda


In [ ]:

# =========================
# 2. 데이터 로드
# =========================
df = pd.read_csv('../data/except_complex.csv', encoding='utf-8-sig')
df_complex = pd.read_csv('../data/complex.csv', encoding='utf-8-sig')

# 필요 없는 장르 제거
df = df[df['장르명'] != '서커스/마술'].reset_index(drop=True)
df_complex = df_complex[df_complex['장르명'] != '서커스/마술'].reset_index(drop=True)
if '세부장르명' in df_complex.columns:
    df_complex = df_complex[df_complex['세부장르명'] != '다원/융복합'].reset_index(drop=True)

In [ ]:
# =========================
# 3. 이름 전처리
# =========================
def split_and_clean(text):
    if pd.isna(text): return []
    names = text.replace(' ', '').split(',')
    names = [re.sub(r'\(.*?\)| 등| 외| 外', '', name).strip() for name in names]
    return [name for name in names if name]

df['출연진내용_standardized'] = df['출연진내용'].apply(split_and_clean)
df['제작진내용_standardized'] = df['제작진내용'].apply(split_and_clean)
df_complex['출연진내용_standardized'] = df_complex['출연진내용'].apply(split_and_clean)
df_complex['제작진내용_standardized'] = df_complex['제작진내용'].apply(split_and_clean)

# 제작사 표준화
df['제작사_standardized'] = df['기획제작사명'].apply(split_and_clean)
df_complex['제작사_standardized'] = df_complex['기획제작사명'].apply(split_and_clean)

In [ ]:


# =========================
# 4. 노드 딕셔너리 구축 (Fix: 중복 제거)
# =========================
node_dict = {}
current_id = 0

# (A) 중복 컬럼 제거
df = df.loc[:, ~df.columns.duplicated()].copy()
df_complex = df_complex.loc[:, ~df_complex.columns.duplicated()].copy()

# (B) original_index & source 보장
df = df.reset_index(drop=True)
df['original_index'] = df.index
df['source'] = 'df'

df_complex = df_complex.reset_index(drop=True)
df_complex['original_index'] = df_complex.index
df_complex['source'] = 'df_complex'

# (C) 합치기
all_performances = pd.concat([df, df_complex], ignore_index=True)

# (1) 인물 노드
for _, row in all_performances.iterrows():
    for name in row['출연진내용_standardized'] + row['제작진내용_standardized']:
        key = f"인물_{name}"
        if key not in node_dict:
            node_dict[key] = current_id; current_id += 1

# (2) 제작사 노드
for _, row in all_performances.iterrows():
    for name in row['제작사_standardized']:
        key = f"제작사_{name}"
        if key not in node_dict:
            node_dict[key] = current_id; current_id += 1

# (3) 공연 노드
for _, row in all_performances.iterrows():
    orig_idx = int(row['original_index'])
    key = f"공연_{row['source']}_{orig_idx}"
    if key not in node_dict:
        node_dict[key] = current_id; current_id += 1

# (4) 장르 노드
genre_names = sorted(list(df['장르명'].apply(lambda x: re.sub(r'\s*\(.*?\)', '', str(x)).strip()).unique()))
for genre in genre_names:
    key = f"장르_{genre}"
    if key not in node_dict:
        node_dict[key] = current_id; current_id += 1

num_nodes = len(node_dict)
print(f"[INFO] 총 노드 수: {num_nodes}")


[INFO] 총 노드 수: 142215


In [ ]:


# =========================
# 5. 엣지 구축 + TF-IDF 가중치 (제작사 포함)
# =========================
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm

edges, edge_weights = [], []

# (A) TF-IDF 코퍼스 생성
corpus = [
    " ".join(row['출연진내용_standardized'] + row['제작진내용_standardized'] + row['제작사_standardized'])
    for _, row in all_performances.iterrows()
]
vectorizer = TfidfVectorizer(token_pattern=r"[^ ]+")
tfidf = vectorizer.fit_transform(corpus)

# (B) 공연별 엣지 구축
for perf_idx, row in tqdm(all_performances.iterrows(), total=len(all_performances), desc="Edges Build"):
    perf_key = f"공연_{row['source']}_{row['original_index']}"
    perf_id = node_dict[perf_key]

    participants = (
        row['출연진내용_standardized'] +
        row['제작진내용_standardized'] +
        row['제작사_standardized']
    )

    for name in participants:
        pkey = f"제작사_{name}" if name in row['제작사_standardized'] else f"인물_{name}"
        if pkey in node_dict:
            pid = node_dict[pkey]
            edges.append([perf_id, pid])
            col_idx = vectorizer.vocabulary_.get(name, None)
            w = float(tfidf[perf_idx, col_idx]) if col_idx is not None else 0.0
            edge_weights.append(w)

# (C) Torch 변환
edges_array = np.array(edges, dtype=np.int64)
edge_index = torch.tensor(edges_array.T, dtype=torch.long)
edge_weight = torch.tensor(edge_weights, dtype=torch.float)

edge_index, edge_weight = to_undirected(edge_index, edge_weight, num_nodes=num_nodes)
edge_index, edge_weight = add_self_loops(edge_index, edge_weight, fill_value=1.0, num_nodes=num_nodes)

edge_index = edge_index.to(device)
edge_weight = edge_weight.to(device)


Edges Build: 100%|██████████| 73711/73711 [00:16<00:00, 4560.45it/s]


In [ ]:

# =========================
# 6. 라벨 (멀티라벨, tqdm 적용)
# =========================
genre_id_labels = {genre: i for i, genre in enumerate(genre_names)}
y = torch.zeros((num_nodes, len(genre_names)), dtype=torch.float).to(device)
train_mask = torch.zeros(num_nodes, dtype=torch.bool).to(device)
predict_mask = torch.zeros(num_nodes, dtype=torch.bool).to(device)

from tqdm import tqdm

print("[INFO] 학습용 라벨 세팅 중...")
for _, row in tqdm(df.iterrows(), total=len(df), desc="Train Mask"):
    perf_key = f"공연_{row['source']}_{row['original_index']}"
    if perf_key in node_dict:
        node_id = node_dict[perf_key]
        genre_name = re.sub(r'\s*\(.*?\)', '', str(row['장르명'])).strip()
        if genre_name in genre_id_labels:
            y[node_id, genre_id_labels[genre_name]] = 1
            train_mask[node_id] = True

print("[INFO] 예측용 라벨 세팅 중...")
hits, misses = 0, 0
for _, row in tqdm(df_complex.iterrows(), total=len(df_complex), desc="Predict Mask"):
    perf_key = f"공연_{row['source']}_{row['original_index']}"
    if perf_key in node_dict:
        predict_mask[node_dict[perf_key]] = True
        hits += 1
    else:
        misses += 1
        if misses < 10:
            print("[MISS]", perf_key)

print(f"[CHECK] 매칭 성공 {hits}, 실패 {misses}")

data = Data(x=None, edge_index=edge_index, y=y).to(device)
data.train_mask = train_mask
data.predict_mask = predict_mask
data.num_nodes = num_nodes

print("GNN Data 객체 구축 완료!")
print(f"총 노드 수: {num_nodes}, 학습용 공연 노드 수: {train_mask.sum()}, 예측용 공연 노드 수: {predict_mask.sum()}")

[INFO] 학습용 라벨 세팅 중...


Train Mask: 100%|██████████| 72612/72612 [00:09<00:00, 7589.88it/s]


[INFO] 예측용 라벨 세팅 중...


Predict Mask: 100%|██████████| 1099/1099 [00:00<00:00, 10005.04it/s]

[CHECK] 매칭 성공 1099, 실패 0
GNN Data 객체 구축 완료!
총 노드 수: 142215, 학습용 공연 노드 수: 72612, 예측용 공연 노드 수: 1099


In [ ]:
# =========================
# 7. 모델 정의 (멀티라벨 GCN)
# =========================
class GCN_MultiLabel(nn.Module):
    def __init__(self, num_nodes, num_genres, embedding_dim=64, hidden_channels=64):
        super().__init__()
        self.node_embedding = nn.Embedding(num_embeddings=num_nodes, embedding_dim=embedding_dim)
        self.conv1 = GCNConv(embedding_dim, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, num_genres)

    def forward(self, edge_index, edge_weight=None):
        x = self.node_embedding(torch.arange(self.node_embedding.num_embeddings, device=edge_index.device))
        x = self.conv1(x, edge_index, edge_weight).relu()
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index, edge_weight)
        return x  # BCEWithLogitsLoss expects raw logits


In [ ]:


# =========================
# 8. Loss + Optimizer (pos_weight 반영)
# =========================
class_counts = y[train_mask].sum(dim=0).cpu().numpy()
pos_weight = torch.tensor((len(y) - class_counts) / (class_counts + 1e-6), dtype=torch.float).to(device)

model = GCN_MultiLabel(num_nodes, len(genre_id_labels)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


# =========================
# 9. 학습 루프
# =========================
epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    out = model(edge_index, edge_weight)
    loss = criterion(out[data.train_mask], y[data.train_mask])
    loss.backward()
    optimizer.step()

    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {loss.item():.4f}")

print("\n모델 학습 완료!")

Epoch 20/100, Train Loss: 1.1337
Epoch 40/100, Train Loss: 0.2162
Epoch 60/100, Train Loss: 0.0725
Epoch 80/100, Train Loss: 0.0332
Epoch 100/100, Train Loss: 0.0216

모델 학습 완료!


In [ ]:

# =========================
# 10. 평가 함수
# =========================
def evaluate_multilabel(y_true, probs, threshold=0.3):
    y_pred_bin = (probs > threshold).astype(int)
    micro_f1 = f1_score(y_true, y_pred_bin, average="micro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred_bin, average="macro", zero_division=0)
    hamming = 1 - hamming_loss(y_true, y_pred_bin)
    return micro_f1, macro_f1, hamming

# =========================
# 11. Validation 평가 (GPU 유지)
# =========================
model.eval()
with torch.no_grad():
    logits = model(edge_index, edge_weight)
    probs = torch.sigmoid(logits)  # ✅ GPU Tensor 그대로 유지

val_idx = torch.where(data.train_mask)[0]  # GPU tensor 인덱스
y_true = y[val_idx].cpu().numpy()
y_pred_bin = (probs[val_idx] > 0.5).int().cpu().numpy()

micro_f1, macro_f1, hamming = evaluate_multilabel(y_true, y_pred_bin, threshold=0.3)
print("\n[Validation 평가 결과]")
print(f"Micro-F1: {micro_f1:.4f}, Macro-F1: {macro_f1:.4f}, Hamming Accuracy: {hamming:.4f}")



[Validation 평가 결과]
Micro-F1: 0.9925, Macro-F1: 0.9474, Hamming Accuracy: 0.9978


In [ ]:
# =========================
# 12. 복합 장르 예측 & 저장 (GPU + tqdm)
# =========================
complex_idx = torch.where(data.predict_mask)[0]  # GPU tensor 인덱스
complex_probs = probs[complex_idx]  # (num_complex, num_genres)

reverse_genre_map = {v: k for k, v in genre_id_labels.items()}
predictions = []

from tqdm import tqdm
for i, node_id in enumerate(tqdm(complex_idx.tolist(), desc="[예측 진행중]")):
    # i번째 공연 확률 벡터에서 top-2
    top2 = torch.topk(complex_probs[i], k=2)
    genres = [reverse_genre_map[j.item()] for j in top2.indices]
    vals = [p.item() for p in top2.values]

    # 공연명 매칭
    perf_key = [k for k, v in node_dict.items() if v == node_id][0]
    perf_name = all_performances.loc[
        all_performances.apply(
            lambda r: f"공연_{r['source']}_{r['original_index']}" == perf_key, axis=1
        ), '공연명'
    ].values[0]

    predictions.append({
        "공연명": perf_name,
        "예측복합장르": f"{genres[0]}+{genres[1]}",
        "장르1": genres[0], "확률1": vals[0],
        "장르2": genres[1], "확률2": vals[1]
    })

pred_df = pd.DataFrame(predictions)
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = f"complex_predictions_{ts}.csv"
pred_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"[저장 완료] {out_path}")
print(pred_df.head())


[예측 진행중]: 100%|██████████| 1099/1099 [09:00<00:00,  2.03it/s]

[저장 완료] complex_predictions_20250913_200517.csv
                              공연명     예측복합장르   장르1       확률1   장르2       확률2
0                      아웃 오브 더 블루  서양음악+한국음악  서양음악  0.999990  한국음악  0.000046
1                           이색콘서트   뮤지컬+서양음악   뮤지컬  1.000000  서양음악  0.000052
2                차세대열전, 이어지음 프로젝트  한국음악+서양음악  한국음악  0.995574  서양음악  0.072073
3         헨젤과 그레텔, 토선생 용궁가다 (패키지)  한국음악+서양음악  한국음악  0.999954  서양음악  0.034011
4  최정주의 아트앤뮤직 큐레이션: 20세기를 빛낸 화가들Ⅱ  서양음악+대중음악  서양음악  0.994409  대중음악  0.155497
